# CallSense — Member 3 | 2. Grounded Digest (E3)
هذه النوت بوك تسترجع نصوصًا من `all_samples.csv` ثم تطلب من نموذج لغوي كتابة ملخص **من المصادر المسترجعة فقط** مع citations بصيغة `[S1]` مرتبطة بـ `audio_id`.

**القيود:** لا توجد timestamps أو speaker IDs في الملف، لذلك الاستشهاد سيكون بمعرّف التسجيل فقط. الملخص المولّد يحتاج مراجعة بشرية؛ لا تعتبريه صحيحًا لمجرد أن النموذج كتبه.

يفضّل تشغيلها على T4 GPU.

## 1) تثبيت المكتبات

In [1]:
!pip -q install sentence-transformers transformers accelerate pandas numpy

## 2) رفع ملف CSV

In [2]:
from google.colab import files
uploaded = files.upload()
CSV_PATH = next(name for name in uploaded if name.lower().endswith(".csv"))

Saving team_transcriptions_final (1).csv to team_transcriptions_final (1).csv


## 3) تحميل البيانات ونموذج الاسترجاع

In [4]:
import pandas as pd
import numpy as np
import torch
import random
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# قراءة الملف الجديد
df = pd.read_csv(CSV_PATH)

# التأكد من الأعمدة المطلوبة
required = {"call_id", "whisper_transcript"}
missing = required - set(df.columns)

if missing:
    raise ValueError(
        f"الأعمدة المطلوبة غير موجودة: {missing}. "
        f"الأعمدة الموجودة: {list(df.columns)}"
    )

# تنظيف النصوص
df = df.dropna(subset=["whisper_transcript"]).copy()
df["transcript"] = df["whisper_transcript"].astype(str).str.strip()

df = df[df["transcript"].ne("")].reset_index(drop=True)

# استخدام call_id كمعرّف للمصدر
df["audio_id"] = df["call_id"].astype(str)

# تحميل نموذج التضمين
EMBED_MODEL = "intfloat/multilingual-e5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(EMBED_MODEL, device=device)

# إنشاء embeddings
embeddings = embedder.encode(
    ["passage: " + x for x in df["transcript"]],
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# دالة البحث عن المكالمات الأكثر صلة
def retrieve(query, k=8):
    q = embedder.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = embeddings @ q
    idx = np.argsort(scores)[::-1][:k]

    return [
        {
            "audio_id": df.iloc[i]["audio_id"],
            "text": df.iloc[i]["transcript"],
            "score": float(scores[i]),
            "dialect": str(df.iloc[i].get("dialect", ""))
        }
        for i in idx
    ]

# تجربة البحث
hits = retrieve("شكاوى العملاء عن الفواتير", k=8)

for i, h in enumerate(hits, 1):
    print(
        f"[S{i}] call_id={h['audio_id']} "
        f"score={h['score']:.3f}\n"
        f"{h['text']}\n"
    )

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[S1] call_id=56 score=0.851
أهلا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ مرحباً وصلتني فاتورة تضم الرسوم تجوال بمبلغ كبير معني لم أسافر إلى خارج البلاد أصلاً أعتذر لكم سأراجع تنفاصل الفاتورة منفضلكم انتظروا قليلاً أراني خدمة التجوال فعلاً لخطقهم قبل شهر أنا لم أفعلها ولا مطلبها هذه تاني مرة أتصل أو عدن المواضف السابق بإلغاء الرسوم لا يظهر لدي في سجيل أي طلب سابق بإلغاء الرسوم كيف لا يظهر لقد أخذت رقم بلاغ وسجلته يبدو أنكم لا تأخذون الشكاوي على محمل الرد أعتذر عن ذلك يمكن أن يرفع طلب مراجعة جديد وستصلكم من نتيجة خيال سبع أيام سبع أيام غير مقبولة أريد إلغاء الرسوم الآن وإلا ألغيت الخض وقدمت شكورة رسمية أريد التواصل مع المشرف

[S2] call_id=3 score=0.845
السلام عليكم معك خدمة ماملكي فقد رأيك ساعدك و عليكم السلام هذه المرة التالثة التي تصل فيها بخصوص المشكلة نفسها أعتقدت جداً إشة المشكلة الغيت خدمت الإنترنت الإضافية قبل شهرين وما زالت الرسوم تظهر لي أكي ظهر لي أنه طلب الإلغاء مسجل فعلاً لكن الرسوم استمرت عندك في كل مرة تقول لي هتحل المشكلة بس في الفاتورة تنزل أتفاها من زعاجك إذا مارت جعتوا

## 4) تحميل نموذج اللغة لتوليد الملخص

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 5) إنشاء ملخص grounded مع مصادر

In [6]:
def make_grounded_digest(query, k=8, max_new_tokens=350):
    sources = retrieve(query, k=k)
    source_block = "\\n\\n".join(
        f"[S{i}] audio_id={h['audio_id']}\\nالنص: {h['text']}"
        for i,h in enumerate(sources, 1)
    )
    system = (
        "أنت مساعد يكتب ملخصًا لمشرف مركز اتصال باللغة العربية. "
        "استخدم المعلومات الموجودة في المصادر فقط. "
        "لا تستنتج سببًا أو نتيجة غير مذكورة. "
        "إذا لم تكفِ المصادر، قل إن المعلومات غير كافية. "
        "اكتب 3 إلى 5 نقاط موجزة، وكل نقطة يجب أن تنتهي باستشهاد مثل [S1]. "
        "لا تخترع أرقام تسجيلات أو استشهادات."
    )
    user = f"موضوع الملخص: {query}\\n\\nالمصادر المتاحة:\\n{source_block}\\n\\nاكتب الملخص."
    messages = [{"role":"system","content":system},{"role":"user","content":user}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output = llm.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    digest = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return digest, sources

digest, sources = make_grounded_digest("أبرز شكاوى العملاء المتعلقة بالفواتير")
print(digest)

ملخص الشكاوى الواردة من العملاء:

1. [S1] و[S2] يشكون عدم وجود إمكانية لإلغاء رسوم التجوال أو الإنترنت التي لم يقوموا بطلبها، كما يشكون أيضاً من تأخير الردود على الشكاوى [S1] و[S2].

2. [S3] يشكو عدم حل مشكلة ضعف خدمة الإنترنت رغم تقديم بلاغات عدة، مما أدى إلى استمرار رسوم الخدمة [S3].

3. [S4] يشكو من تكرار الرسائل التسويقية وعدم استجابتها، بالإضافة إلى تأخير الردود على الشكاوى [S4].

4. [S5] يشكو من تأخير إرسال فاتورة الشهر [S5].

5. [S6] يشكو من الرسائل التسويقية المستمرة رغم إيقافها [S6].

6. [S7] يشكو من عدم استجابة الشركة لطلبات الغاء الاشتراك، مما أدى إلى دفع رسوم بدون خدمات [S7].

7. [S8] يشكو من تأخير إرسال فاتورة الشهر، مما أدى إلى عدم قدرته على دفعها [S8].


## 6) التحقق الآلي من أن الاستشهادات تشير إلى مصادر موجودة

In [7]:
import re
def check_citations(digest, sources):
    valid = {f"S{i}" for i in range(1, len(sources)+1)}
    found = re.findall(r"\\[(S\\d+)\\]", digest)
    invalid = sorted(set(found) - valid)
    missing = not found
    return {
        "citations_found": found,
        "invalid_citations": invalid,
        "has_at_least_one_citation": not missing,
        "citation_ids_to_audio": {
            f"S{i}": h["audio_id"] for i,h in enumerate(sources,1)
        }
    }

check = check_citations(digest, sources)
print(check)

{'citations_found': [], 'invalid_citations': [], 'has_at_least_one_citation': False, 'citation_ids_to_audio': {'S1': '56', 'S2': '3', 'S3': '57', 'S4': '51', 'S5': '31', 'S6': '33', 'S7': '22', 'S8': '24'}}


## 7) عرض النصوص التي استند إليها الملخص وحفظه

In [9]:
# عرض المصادر التي استخدمها التلخيص
for i, h in enumerate(sources, 1):
    print(
        f"[S{i}] -> call_id={h['audio_id']} "
        f"| dialect={h.get('dialect', '')}"
    )
    print(h["text"], "\n")

# حفظ التلخيص في ملف نصي
with open("digest.txt", "w", encoding="utf-8") as f:
    f.write(digest)

# تحميل ملف التلخيص
from google.colab import files
files.download("digest.txt")

[S1] -> call_id=56 | dialect=MSA
أهلا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ مرحباً وصلتني فاتورة تضم الرسوم تجوال بمبلغ كبير معني لم أسافر إلى خارج البلاد أصلاً أعتذر لكم سأراجع تنفاصل الفاتورة منفضلكم انتظروا قليلاً أراني خدمة التجوال فعلاً لخطقهم قبل شهر أنا لم أفعلها ولا مطلبها هذه تاني مرة أتصل أو عدن المواضف السابق بإلغاء الرسوم لا يظهر لدي في سجيل أي طلب سابق بإلغاء الرسوم كيف لا يظهر لقد أخذت رقم بلاغ وسجلته يبدو أنكم لا تأخذون الشكاوي على محمل الرد أعتذر عن ذلك يمكن أن يرفع طلب مراجعة جديد وستصلكم من نتيجة خيال سبع أيام سبع أيام غير مقبولة أريد إلغاء الرسوم الآن وإلا ألغيت الخض وقدمت شكورة رسمية أريد التواصل مع المشرف 

[S2] -> call_id=3 | dialect=Saudi
السلام عليكم معك خدمة ماملكي فقد رأيك ساعدك و عليكم السلام هذه المرة التالثة التي تصل فيها بخصوص المشكلة نفسها أعتقدت جداً إشة المشكلة الغيت خدمت الإنترنت الإضافية قبل شهرين وما زالت الرسوم تظهر لي أكي ظهر لي أنه طلب الإلغاء مسجل فعلاً لكن الرسوم استمرت عندك في كل مرة تقول لي هتحل المشكلة بس في الفاتورة تنزل أتفاها من زعاجك إ

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8) تجربة baseline بدون استرجاع (اختياري)
للمقارنة، يمكن توليد رد من النموذج بدون تزويده بمصادر. لا تستخدمي هذا الرد كملخص موثّق؛ هو baseline فقط.

In [10]:
def llm_only_baseline(query, max_new_tokens=250):
    messages = [
        {"role":"system","content":"اكتب ردًا موجزًا بالعربية. لا تدّعِ أنك اطلعت على بيانات مكالمات فعلية."},
        {"role":"user","content":f"اكتب ملخصًا عامًا حول: {query}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(llm_only_baseline("أبرز مشاكل الفواتير في مركز الاتصال"))

في مركز الاتصال، بعض المشاكل الشائعة التي قد تواجهها عند معالجة الفواتير تشمل:

1. الأخطاء التقنية: قد تحدث مشاكل في النظام أو البرامج المستخدمة لمعالجة الفواتير، مما يؤدي إلى أخطاء في التحصيل أو التسليم.

2. عدم دقة البيانات: قد تظهر بيانات غير دقيقة في الفواتير، مثل الأرقام الخاطئة أو التواريخ المبهمة، مما يسبب صعوبات في التحقق من الدفع.

3. تأخير في تسليم الفواتير: قد يحدث تأخير في إرسال الفواتير للمشتركين، مما قد يؤدي إلى تأخر في الدفع.

4. مشكلات في التواصل: قد يكون هناك صعوبة في التواصل بين الموظفين المسؤولين عن معالجة الفواتير والمشتركين، مما يؤدي إلى سوء فهم ومشكلات في الدفع.

5. قضايا معقدة: قد تواجه الفواتير بعض القضايا المعقدة مثل التغييرات القانونية أو التحويلات المالية، والتي قد تتطلب وقتًا إ


## ملاحظات مهمة
- صحة الاستشهاد لا تعني تلقائيًا صحة المعلومة؛ راجعي كل نقطة مقابل النص المشار إليه.
- فحص الاستشهاد في هذه النوت بوك يتحقق من وجود رمز `[S#]` فقط، ولا يثبت أن المصدر يدعم الادعاء.
- أرقام Recall@5 وfaithfulness وcitation accuracy يجب حسابها من أسئلة ووسوم مرجعية حقيقية، ولا تستخدمي نتائج تجريبية على أنها نتائج نهائية.